# Grandmaster Pipeline: Scaled NAFNet + Perceptual Loss + EMA

This notebook trains a **Heavyweight NAFNet** optimized specifically for winning Kaggle metrics. It features aggressive data augmentations (D4 Symmetry + CutMix), a VGG-19 Perceptual Loss for high-frequency hallucination, Cosine Annealing, and Exponential Moving Average (EMA) weight stabilization.

### Optimizations (Fast Training):
- **Automatic Mixed Precision (AMP):** Uses FP16 to train 2x faster with half the memory.
- `pin_memory=True` for faster CPU-to-GPU data transfer.
- Tuned Channel count (48) for optimal Speed/Accuracy tradeoff.

In [ ]:
# Ensure required libraries are installed
!pip install -q onnx onnxruntime-gpu onnxscript scikit-image lpips torchvision

import os
import glob
import time
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim_metric
import lpips

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, random_split
import onnxruntime as ort

# --- CONFIGURATION ---
DRY_RUN = True       # Set to False to train fully
EPOCHS = 2 if DRY_RUN else 40
BATCH_SIZE = 8       
LEARNING_RATE = 4e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. Dataset, DataLoader & D4 Augmentations

In [ ]:
def find_dataset_paths():
    search_paths = ['/kaggle/input', '.']
    train_gt, train_noisy, test_noisy = None, None, None
    for base in search_paths:
        if not os.path.exists(base): continue
        for root, dirs, files in os.walk(base):
            if not any(f.endswith('.npy') for f in files): continue
            dir_name = os.path.basename(root)
            if dir_name == 'GT' and 'train' in root.lower(): train_gt = root
            elif dir_name == 'NoisyLR' and 'train' in root.lower(): train_noisy = root
            elif dir_name == 'NoisyLR' and 'test' in root.lower(): test_noisy = root
    return train_gt, train_noisy, test_noisy

TRAIN_GT_DIR, TRAIN_DEGRADED_DIR, TEST_DEGRADED_DIR = find_dataset_paths()

class KLADataset(Dataset):
    def __init__(self, gt_dir, deg_dir, is_train=False):
        self.gt_files = sorted(glob.glob(os.path.join(gt_dir, '*.npy')))
        self.deg_files = sorted(glob.glob(os.path.join(deg_dir, '*.npy')))
        self.min_len = min(len(self.gt_files), len(self.deg_files))
        self.is_train = is_train
        
    def __len__(self):
        return self.min_len
    
    def __getitem__(self, idx):
        gt = np.load(self.gt_files[idx])
        deg = np.load(self.deg_files[idx])
        
        gt = torch.tensor(gt, dtype=torch.float32).unsqueeze(0)
        deg = torch.tensor(deg, dtype=torch.float32).unsqueeze(0)
        
        # D4 Symmetry Augmentation
        if self.is_train:
            if random.random() < 0.5:
                gt = torch.flip(gt, [-1])
                deg = torch.flip(deg, [-1])
            if random.random() < 0.5:
                gt = torch.flip(gt, [-2])
                deg = torch.flip(deg, [-2])
            k = random.randint(0, 3)
            if k > 0:
                gt = torch.rot90(gt, k, [-2, -1])
                deg = torch.rot90(deg, k, [-2, -1])
                
        return deg, gt

full_dataset = KLADataset(TRAIN_GT_DIR, TRAIN_DEGRADED_DIR, is_train=False)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# Override the train_ds dataset class to enable augmentations
train_ds.dataset.is_train = True

full_train_len = len(train_ds)
if DRY_RUN:
    # Increased to 80 images (10 batches) to allow for warmup before timing
    train_ds = torch.utils.data.Subset(train_ds, range(80))
    val_ds = torch.utils.data.Subset(val_ds, range(16))

# OPTIMIZATION: pin_memory=True speeds up CPU -> GPU transfers
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


## 2. Model: Scaled NAFNet

In [ ]:
class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c * 2, 1)
        self.conv2 = nn.Conv2d(c * 2, c * 2, 3, padding=1, groups=c * 2)
        self.sg = SimpleGate()
        
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, c, 1)
        )
        self.conv3 = nn.Conv2d(c, c, 1)
        self.norm1 = nn.GroupNorm(1, c)
        
        self.conv4 = nn.Conv2d(c, c * 2, 1)
        self.conv5 = nn.Conv2d(c, c, 1)
        self.norm2 = nn.GroupNorm(1, c)
        
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)))
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)))

    def forward(self, x):
        inp = x
        x = self.norm1(x)
        
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        
        y = inp + x * self.beta
        
        x = self.norm2(y)
        x = self.conv4(x)
        x = self.sg(x)
        x = self.conv5(x)
        
        return y + x * self.gamma

class ScaledNAFNet(nn.Module):
    def __init__(self):
        super().__init__()
        c = 48
        self.intro = nn.Conv2d(1, c, 3, padding=1)
        
        self.enc1 = nn.Sequential(*[NAFBlock(c) for _ in range(2)])
        self.down = nn.Conv2d(c, c * 2, 2, stride=2)
        self.enc2 = nn.Sequential(*[NAFBlock(c * 2) for _ in range(2)])
        
        self.mid = nn.Sequential(*[NAFBlock(c * 2) for _ in range(2)])
        
        self.up = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = nn.Sequential(*[NAFBlock(c) for _ in range(2)])
        
        self.ending = nn.Conv2d(c, 4, 3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(2)

    def forward(self, x):
        x1 = self.intro(x)
        x1 = self.enc1(x1)
        x2 = self.down(x1)
        x2 = self.enc2(x2)
        x2 = self.mid(x2)
        x_up = self.up(x2)
        x_up = x_up + x1
        x_out = self.dec1(x_up)
        out = self.ending(x_out)
        return self.pixel_shuffle(out)

model = ScaledNAFNet().to(DEVICE)

# Exponential Moving Average (EMA)
class EMA():
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
    def register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

ema = EMA(model, decay=0.999)
ema.register()

## 3. Losses, Augmentations & Grandmaster Training Loop (with AMP)

In [ ]:
# Edge Loss
class EdgeAwareLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()
        kernel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        kernel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('kernel_x', kernel_x)
        self.register_buffer('kernel_y', kernel_y)
        
    def forward(self, pred, target):
        px, py = nn.functional.conv2d(pred, self.kernel_x, padding=1), nn.functional.conv2d(pred, self.kernel_y, padding=1)
        tx, ty = nn.functional.conv2d(target, self.kernel_x, padding=1), nn.functional.conv2d(target, self.kernel_y, padding=1)
        return self.l1(px, tx) + self.l1(py, ty)

# VGG Perceptual Loss
class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        self.features = nn.Sequential(*list(vgg.children())[:26]).eval().to(DEVICE)
        for param in self.features.parameters(): param.requires_grad = False
        self.l1 = nn.L1Loss()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))
        
    def forward(self, pred, target):
        pred_rgb, target_rgb = pred.repeat(1, 3, 1, 1), target.repeat(1, 3, 1, 1)
        pred_norm = (pred_rgb - self.mean) / self.std
        target_norm = (target_rgb - self.mean) / self.std
        return self.l1(self.features(pred_norm), self.features(target_norm))

l1_loss_fn = nn.L1Loss()
edge_loss_fn = EdgeAwareLoss().to(DEVICE)
vgg_loss_fn = VGGLoss().to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# OPTIMIZATION: Automatic Mixed Precision (AMP) Scaler
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

lpips_metric = lpips.LPIPS(net='alex').to(DEVICE)
lpips_metric.eval()

def apply_cutmix(deg, gt):
    if random.random() > 0.5: return deg, gt
    lam = np.random.beta(1.0, 1.0)
    bs = deg.size(0)
    index = torch.randperm(bs).to(DEVICE)
    H, W = 128, 128
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bx1, by1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bx2, by2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    deg[:, :, by1:by2, bx1:bx2] = deg[index, :, by1:by2, bx1:bx2]
    gt[:, :, by1*2:by2*2, bx1*2:bx2*2] = gt[index, :, by1*2:by2*2, bx1*2:bx2*2]
    return deg, gt

def calculate_metrics(pred, gt):
    p, g = pred.cpu().detach().numpy().clip(0, 1), gt.cpu().detach().numpy().clip(0, 1)
    psnr_total, ssim_total, bs = 0, 0, p.shape[0]
    for i in range(bs):
        pi, gi = p[i, 0], g[i, 0]
        mse = np.mean((pi - gi) ** 2)
        psnr_total += 100 if mse == 0 else 20 * math.log10(1.0 / math.sqrt(mse))
        ssim_total += ssim_metric(gi, pi, data_range=1.0)
    with torch.no_grad():
        lpips_batch_sum = lpips_metric(pred.clip(0,1)*2-1, gt.clip(0,1)*2-1).sum().item()
    return psnr_total, ssim_total, lpips_batch_sum

def get_fft_magnitude(img):
    f_transform = np.fft.fft2(img)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1e-8)
    return magnitude_spectrum

def plot_diagnostics(deg, gt, pred):
    deg, gt, pred = deg[0,0].cpu().numpy(), gt[0,0].cpu().numpy(), pred[0,0].cpu().detach().numpy()
    error_map = np.abs(gt - pred)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0,0].imshow(deg, cmap='gray'); axes[0,0].axis('off')
    axes[0,1].imshow(pred, cmap='gray'); axes[0,1].axis('off')
    axes[0,2].imshow(gt, cmap='gray'); axes[0,2].axis('off')
    axes[1,0].imshow(error_map, cmap='hot'); axes[1,0].axis('off')
    axes[1,1].imshow(get_fft_magnitude(pred), cmap='magma'); axes[1,1].axis('off')
    axes[1,2].imshow(get_fft_magnitude(gt), cmap='magma'); axes[1,2].axis('off')
    plt.show()

best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    # Warmup timing logic for accurate DRY_RUN estimation
    warmup_batches = 3
    timed_batches = 0
    start_time_after_warmup = None
    
    for i, (deg, gt) in enumerate(train_pbar):
        if DRY_RUN and i == warmup_batches:
            start_time_after_warmup = time.time()
            
        deg, gt = deg.to(DEVICE), gt.to(DEVICE)
        deg, gt = apply_cutmix(deg, gt)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred = model(deg)
            loss = l1_loss_fn(pred, gt) + 0.1 * edge_loss_fn(pred, gt) + 0.05 * vgg_loss_fn(pred, gt)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update()
        
        train_loss += loss.item()
        train_pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
        if DRY_RUN and i >= warmup_batches:
            timed_batches += 1
            
    # Accurate Dry Run Estimation (Ignoring PyTorch compilation overhead)
    if DRY_RUN and epoch == 0 and start_time_after_warmup is not None:
        clean_train_time = time.time() - start_time_after_warmup
        images_processed = timed_batches * BATCH_SIZE
        if clean_train_time > 0:
            images_per_sec = images_processed / clean_train_time
            est_full_epoch_time = full_train_len / images_per_sec
            print(f"\n[DRY RUN DETECTED] Accurate estimate for FULL epoch: ~{est_full_epoch_time/60:.1f} minutes.")
        if torch.cuda.is_available():
            mem_gb = torch.cuda.max_memory_allocated() / (1024**3)
            print(f"[DRY RUN DETECTED] Peak GPU Memory Used: {mem_gb:.2f} GB")
        
    scheduler.step()
    
    ema.apply_shadow()
    model.eval()
    val_loss, val_psnr, val_ssim, val_lpips, num_val_samples = 0, 0, 0, 0, 0
    with torch.no_grad():
        for i, (deg, gt) in enumerate(val_loader):
            deg, gt = deg.to(DEVICE), gt.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                pred = model(deg)
                v_loss = l1_loss_fn(pred, gt).item()
            val_loss += v_loss
            batch_psnr, batch_ssim, batch_lpips = calculate_metrics(pred, gt)
            val_psnr += batch_psnr; val_ssim += batch_ssim; val_lpips += batch_lpips
            num_val_samples += deg.size(0)
            if i == 0: plot_diagnostics(deg, gt, pred)
            
    val_loss /= len(val_loader)
    val_psnr /= num_val_samples; val_ssim /= num_val_samples; val_lpips /= num_val_samples
    
    print(f"Epoch {epoch+1} | Val L1: {val_loss:.4f} | PSNR: {val_psnr:.2f} | SSIM: {val_ssim:.4f} | LPIPS: {val_lpips:.4f}\n")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_nafnet_ema.pt')
        
    ema.restore()


## 4. Export to ONNX

In [ ]:
def export_and_benchmark():
    if not os.path.exists('best_nafnet_ema.pt'): return
    print("--- Exporting to ONNX ---")
    model.load_state_dict(torch.load('best_nafnet_ema.pt'))
    model.eval()
    dummy_input = torch.randn(1, 1, 128, 128).to(DEVICE)
    onnx_path = "nafnet_grandmaster.onnx"
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.onnx.export(model, dummy_input, onnx_path, export_params=True, opset_version=17, do_constant_folding=True, input_names=['input'], output_names=['output'])
    print(f"Successfully exported to {onnx_path}")
    
    print("\n--- Benchmarking ONNX Runtime ---")
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    ort_session = ort.InferenceSession(onnx_path, providers=providers)
    
    test_files = glob.glob(os.path.join(TEST_DEGRADED_DIR, '*.npy'))
    if not test_files:
        print("No test files found for benchmarking.")
        return
        
    num_test = min(20, len(test_files))
    latencies = []
    
    for i in range(num_test):
        deg_img = np.load(test_files[i]).astype(np.float32)
        deg_input = np.expand_dims(np.expand_dims(deg_img, axis=0), axis=0)
        
        start_time = time.perf_counter()
        ort_inputs = {ort_session.get_inputs()[0].name: deg_input}
        ort_outs = ort_session.run(None, ort_inputs)
        latencies.append(time.perf_counter() - start_time)
        
    avg_time_ms = np.mean(latencies[1:]) * 1000
    print(f"ONNX Inference Benchmark: ~{avg_time_ms:.2f} ms per image on {ort_session.get_providers()[0]}")
    
export_and_benchmark()